# V2

In [1]:
!pip install -q selenium chromedriver_autoinstaller webdriver_manager

In [2]:
!apt-get update
!apt-get install chromium-browser

Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:6 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:8 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading packag

In [3]:
import pandas as pd
import numpy as np

import os
import re
import time
import math
import pandas as pd
from typing import List, Dict, Optional
from selenium import webdriver
from selenium.webdriver.chrome.service import Service as ChromeService
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.common.exceptions import (
    NoSuchElementException, TimeoutException, StaleElementReferenceException
)
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

from google.colab import drive
drive.mount('/content/gdrive')

Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).


In [4]:
def setup_driver(headless: bool = False, window_size: str = "1200, 900") -> webdriver.Chrome:
    chrome_options = Options()
    chrome_options.add_argument("--headless")
    # chrome_options.add_argument("--disable-gpu")
    # chrome_options.add_argument(f"--window-size={window_size}")
    chrome_options.add_argument("--disable-dev-shm-usage")
    chrome_options.add_argument("--no-sandbox")
    # keep default user agent / do not spoof aggressively
    # service = ChromeService(ChromeDriverManager().install())
    driver = webdriver.Chrome(options=chrome_options)
    driver.set_page_load_timeout(60)
    return driver

In [5]:
def wait_click(driver, by, sel, timeout=60):
    el = WebDriverWait(driver, timeout).until(
        EC.element_to_be_clickable((by, sel))
    )
    time.sleep(0.5)
    driver.execute_script("arguments[0].click();", el)
    time.sleep(0.5)
    return el

def find_reviews_scrollbox(driver):
    # Try stable ARIA/roles first
    el = driver.execute_script("""
      // Known stable containers
      const byAria = document.querySelector('div[aria-label="Google reviews"]');
      if (byAria) return byAria;

      // Region that contains review list
      const regions = Array.from(document.querySelectorAll('div[role="region"], section[role="region"]'));
      for (const r of regions){
        if (r.querySelector('div.jftiEf')) return r;
      }

      // Fallback: parent of a review card
      const card = document.querySelector('div.jftiEf, div.TFQHme, div.AyRUI');
      if (!card) return document.scrollingElement || document.documentElement || document.body;

      function isScrollable(e){
        if (!e) return false;
        const cs = getComputedStyle(e);
        const oy = cs.overflowY;
        return (oy === 'auto' || oy === 'scroll') && e.scrollHeight > e.clientHeight;
      }
      let node = card;
      while (node && node !== document.body && node !== document.documentElement){
        node = node.parentElement;
        if (isScrollable(node)) return node;
      }
      return document.scrollingElement || document.documentElement || document.body;
    """)
    return el

In [6]:
def scroll_to_bottom_until_idle(driver, scrollable, idle_ms=2500, max_ms=1_000_000):
    """
    Scrolls to bottom until the container height is stable for idle_ms,
    or until max_ms is reached. Adds 0.5 sec wait between scroll steps.
    """
    driver.set_script_timeout((max_ms // 1000) + 5)
    driver.execute_async_script("""
      const sc = arguments[0];
      const idle = arguments[1];
      const maxms = arguments[2];
      const done = arguments[3];

      let lastH = -1;
      let lastGrowthAt = performance.now();
      const t0 = performance.now();

      function tick(){
        try {
          sc.scrollTop = sc.scrollHeight; // jump to bottom
        } catch(e) {}

        const h = sc.scrollHeight;
        const now = performance.now();
        if (h !== lastH){
          lastH = h;
          lastGrowthAt = now;
        }
        if ((now - lastGrowthAt) >= idle || (now - t0) >= maxms){
          return done(h);
        }
        // wait 0.5s before next frame to let lazy-load fetch more reviews
        setTimeout(() => requestAnimationFrame(tick), 1000);
      }
      tick();
    """, scrollable, idle_ms, max_ms)


In [7]:
def expand_all_truncated_reviews(driver):
    # Click "More"/"Read more" buttons so you capture full text
    try:
        driver.execute_script("""
          const btns = Array.from(document.querySelectorAll('button, span'));
          for (const b of btns){
            const t = (b.innerText || '').toLowerCase();
            if (t === 'more' || t === 'read more' || t.startsWith('more'))
              b.click();
          }
        """)
    except Exception:
        pass

In [8]:
def extract_reviews_batch(driver):
    # Return array of dicts from the DOM in one go
    return driver.execute_script("""
      const cards = document.querySelectorAll("div.jftiEf");
      const out = [];
      for (const rb of cards){
        const getText = sel => {
          const el = rb.querySelector(sel);
          return el ? el.textContent.trim() : null;
        };
        const getRating = () => {
          const s = rb.querySelector("span[aria-label*='Rated'], span[aria-label*='stars'], span[aria-label*='out of']");
          if (!s) return null;
          const t = s.getAttribute("aria-label") || s.textContent || "";
          const m = t.match(/(\\d+(?:\\.\\d+)?)/);
          return m ? parseFloat(m[1]) : null;
        };
        out.push({
          author: getText(".d4r55"),
          date:   getText(".rsqaWe"),
          text:   getText(".wiI7pd"),
          rating: getRating()
        });
      }
      return out;
    """)


In [9]:
def fetch_reviews_for_place(place_id: str,
                            driver: webdriver.Chrome,
                            max_reviews: int = 1_000_000,
                            max_scroll_ms: int = 1_000_000,
                            idle_ms: int = 2500,
                            open_timeout: int = 2500) -> List[Dict]:
    url = f"https://www.google.com/maps/place/?q=place_id:{place_id}"
    driver.get(url)
    # time.sleep(0.5)
    # Open the Reviews panel quickly (avoid broad XPaths, use a few robust tries)
    try:
        # common pattern: a button with inner text 'Reviews'
        wait_click(driver, By.XPATH, "//button[.//span[contains(., 'Reviews')] or contains(., 'Reviews')]", timeout=open_timeout)
    except TimeoutException:
        # fallback: aaria label variants
        wait_click(driver, By.XPATH, "//button[contains(@aria-label,'review') or contains(., 'review')]", timeout=open_timeout)

    # Find the scroll container and drive it from JS (no sleeps)
    scrollbox = find_reviews_scrollbox(driver)
    scroll_to_bottom_until_idle(driver, scrollbox, idle_ms=idle_ms, max_ms=max_scroll_ms)

    # One batched extraction from DOM
    rows = extract_reviews_batch(driver)

    # Dedupe & trim to max_reviews
    seen = set()
    out = []
    for r in rows:
        key = (r.get("author"), r.get("date"), r.get("text"), r.get("rating"))
        if key in seen:
            continue
        seen.add(key)
        out.append({
            "place_id": place_id,
            "author": r.get("author"),
            "rating": r.get("rating"),
            "date": r.get("date"),
            "text": r.get("text")
        })
        if len(out) >= max_reviews:
            break

    return out

In [10]:
def collect_reviews_from_excel_selenium(excel_path: str,
                                        start_row: int,
                                        # output_csv: str,
                                        place_id_col: str,
                                        place_name_col: str,
                                        max_reviews_per_place: int = 1_000_000,
                                        headless: bool = True):
    df = pd.read_excel(excel_path).iloc[start_row:, :]
    if place_id_col is None:
        raise ValueError("Could not find a place_id column in the Excel file.")

    driver = setup_driver(headless=headless)

    try:
        unique_ids = df[place_id_col].dropna().astype(str).unique()
        for i, pid in enumerate(unique_ids, 1):
            start_time = time.time()  # <-- start timer

            results = []
            place_name = df.loc[df['id']==pid, place_name_col].values[0]
            safe_place_name = re.sub(r'[<>:"/\\|?*]', '_', place_name)
            place_row = df.loc[df['id']==pid, 'index'].values[0]
            filename = f"{place_row}. {safe_place_name}.xlsx"
            filepath = os.path.join("/content/gdrive/My Drive/PhD/reviews_part1", filename)

            print(f"[{i}/{len(unique_ids)}] Scraping place_id={pid} ...")
            try:
                reviews = fetch_reviews_for_place(pid, driver)
                if not reviews:
                    results.append({"place_id": pid, "text": "no_reviews_or_could_not_open"})
                else:
                    for r in reviews:
                        results.append(r)
            except Exception as e:
                print(f"[ERROR] place {pid}: {e}")
                results.append({"place_id": pid, "error": str(e)})
            # small delay between places to be polite
            # time.sleep(1.0)
            out_df = pd.DataFrame(results)
            out_df.to_excel(filepath)
            print(f"Iteration runtime: {time.time() - start_time:.2f} seconds")
            print(filename)

    finally:
        driver.quit()

    print('DONE!!!')

In [ ]:
if __name__ == "__main__":
    EXCEL_PATH = "/content/gdrive/My Drive/PhD/reviews_places_australia_sa1_part1.xlsx"
    MAX_REVIEWS = 1_000_000
    HEADLESS = False
    START_ROW = 0

    collect_reviews_from_excel_selenium(
        excel_path=EXCEL_PATH,
        start_row=START_ROW,
        # output_csv=OUTPUT_CSV,
        place_id_col='id',
        place_name_col='place_name',
        max_reviews_per_place=MAX_REVIEWS,
        headless=HEADLESS
    )

[1/19402] Scraping place_id=ChIJ3S-JXmauEmsRUcIaWtf4MzE ...
